# Multi-Class Image Classification Using CNN

A practical implementation of a Convolutional Neural Network (CNN) using PyTorch. This notebook uses the CIFAR-10 dataset as the image-classification benchmark.

## 1. Import libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision import transforms

## 2. Load and preprocess the image dataset (CIFAR-10)

CIFAR-10 is the dataset used in this implementation. Images are converted to tensors and normalized across their three RGB channels.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

CIFAR10.url = (
    'https://huggingface.co/datasets/MIT-OL-AI-D/cifar-10-python/'
    'resolve/c6e562a/cifar-10-python.tar.gz'
)

train_dataset = CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## 3. Build the CNN architecture

Three convolutional blocks extract visual features. Max pooling reduces the spatial dimensions, and fully connected layers produce 10 raw class scores (logits). ReLU is used in the hidden layers. The final layer does not need a sigmoid or softmax when using `CrossEntropyLoss`: PyTorch applies the required log-softmax calculation internally.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(4 * 4 * 128, 256),
            nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, images):
        features = self.conv_layers(images)
        features = features.view(features.size(0), -1)
        return self.fc_layers(features)

model = CNN()

## 4. Configure training

Cross-entropy loss is used for multi-class classification, with Adam as the optimizer.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## 5. Train the model

For each batch, perform a forward pass, calculate loss, backpropagate gradients, and update the model parameters.

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_training_loss += loss.item()

    average_loss = epoch_training_loss / len(train_loader)
    print(f'Epoch {epoch + 1}/{epochs} - Training Loss: {average_loss:.4f}')

## 6. Evaluate the model

In [ ]:
model.eval()
correct_predictions = 0
total_predictions = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        predicted_classes = outputs.argmax(dim=1)
        correct_predictions += (predicted_classes == labels).sum().item()
        total_predictions += labels.size(0)

test_accuracy = 100.0 * correct_predictions / total_predictions
print(f'Test Accuracy: {test_accuracy:.2f}%')

## Recorded result

The original executed run recorded **74.71% test accuracy** after 10 epochs. A fresh run may vary because initialization and data shuffling are stochastic.